# Energy_indicators

Final energy by carrier and service, primary energy and CO₂ of a project, with the gallery example 09 (air-to-water heat pump with a backup boiler, two circuits, fan coils, two offices of 48 m² defined by their loads).

In [ ]:
import json, os
import opensimula as osm
from opensimula import gui

os.makedirs("energy_example", exist_ok=True)
gui.copy_examples("energy_example")
os.chdir("energy_example")

Example 09 plus one component: `Energy_indicators` with the default parameters (every consumer and zone of the project, factors of the RITE for the peninsula).

In [ ]:
project = json.load(open("09_water_plant.json", encoding="utf-8"))
project["components"].append({
    "type": "Energy_indicators",
    "name": "energy",
    "conversion_factors": "PENINSULA",
})
sim = osm.Simulation()
pro = sim.new_project("09")
pro.read_dict(project)
pro.simulate()

## The report

Default hypotheses: pumps (`WATER_TRANSPORT`) and fans (`AIR_TRANSPORT`) as their own services.

In [ ]:
energy = pro.component("energy")
energy.report()

Pumps and fans assigned to heating or cooling by the mode of each hour (what runs without load stays as transport). Only the water, only the air, or both.

In [ ]:
energy.report(water_transport="BY_MODE", air_transport="BY_MODE")

The tables are pandas DataFrames, and `summary` a dict:

In [ ]:
r = energy.report()
r.summary

In [ ]:
r.monthly_carriers

## Hourly variables

`electricity`, `natural_gas`, `ambient`, `thermal` (W), one per service (`heating`, `cooling`, `water_transport`...), `primary_ren`, `primary_nren`, `primary_total` (W) and `CO2` (kg/h). They can be plotted in the results viewer.

In [ ]:
energy.variable_dataframe(units=True, frequency="monthly", value="sum")

In [ ]:
pro.results()

## Heat without equipment

Example 05 has an ideal system: its heat has no generator (`THERMAL`), out of the totals until reference generators are given. The office has lighting and equipment (electric by default, `equipment_carrier`).

In [ ]:
ideal = json.load(open("05_ideal_system.json", encoding="utf-8"))
ideal["components"].append({"type": "Energy_indicators", "name": "energy"})
pro5 = sim.new_project("05")
pro5.read_dict(ideal)
pro5.simulate()
pro5.component("energy").report()

In [ ]:
pro5.component("energy").report(reference_heating=("NATURAL_GAS", 0.92), reference_cooling=("ELECTRICITY", 2.6))